# 01 数据下载与预处理

本notebook负责从akshare获取股票数据，计算收益率，并进行基本的描述性统计分析。

## 1. 导入必要的库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import akshare as ak
import warnings
from datetime import datetime, date
from scipy import stats
import os

# 设置中文字体和图形样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_style("whitegrid")
warnings.filterwarnings('ignore')

print("库导入成功")
print(f"当前工作目录：{os.getcwd()}")

## 2. 定义股票列表和数据参数

In [ ]:
# 定义股票代码和基本信息
stocks_info = {
    '000001': {'name': '平安银行', 'industry': '银行'},
    '000858': {'name': '五粮液', 'industry': '消费'},
    '000002': {'name': '万科A', 'industry': '地产'},
    '300059': {'name': '东方财富', 'industry': '金融科技'},
    '600036': {'name': '招商银行', 'industry': '银行'}
}

stock_codes = list(stocks_info.keys())
print("选定股票列表：")
for code, info in stocks_info.items():
    print(f"{code}: {info['name']} ({info['industry']})")

# 数据时间范围
start_date = "20190101"
end_date = datetime.now().strftime("%Y%m%d")
print(f"\n数据时间范围：{start_date} ~ {end_date}")

# 无风险利率设定（年化2.5%）
rf_annual = 0.025
rf_daily = rf_annual / 252  # 转换为日收益率
print(f"无风险利率：年化{rf_annual*100:.1f}%, 日化{rf_daily*10000:.2f}bp")

## 3. 数据下载函数

In [ ]:
def download_stock_data(symbol, symbol_name, start_date, end_date, is_index=False):
    """
    下载股票或指数数据
    
    Parameters:
    - symbol: 股票代码或指数代码
    - symbol_name: 股票名称
    - start_date: 开始日期
    - end_date: 结束日期  
    - is_index: 是否为指数
    
    Returns:
    - DataFrame: 包含日期和收盘价的数据
    """
    try:
        if is_index:
            # 下载指数数据
            print(f"正在下载指数数据：{symbol_name} ({symbol})")
            data = ak.index_zh_a_hist(symbol=symbol, period="daily", start_date=start_date, end_date=end_date)
        else:
            # 下载股票数据（后复权）
            print(f"正在下载股票数据：{symbol_name} ({symbol})")
            data = ak.stock_zh_a_hist(symbol=symbol, period="daily", start_date=start_date, end_date=end_date, adjust="hfq")
        
        if data.empty:
            print(f"警告：{symbol_name} 数据为空")
            return None
            
        # 统一列名和数据格式
        data['日期'] = pd.to_datetime(data['日期'])
        data = data[['日期', '收盘']].copy()
        data.columns = ['date', symbol]
        data.set_index('date', inplace=True)
        data[symbol] = pd.to_numeric(data[symbol], errors='coerce')
        
        print(f"成功下载 {symbol_name}，数据量：{len(data)} 条")
        return data
        
    except Exception as e:
        print(f"下载 {symbol_name} 数据失败：{e}")
        return None

## 4. 下载所有数据

In [ ]:
# 下载沪深300指数数据
print("=== 开始下载基准指数数据 ===")
hs300_data = download_stock_data("000300", "沪深300", start_date, end_date, is_index=True)

# 下载股票数据
print("\n=== 开始下载股票数据 ===")
stock_data = {}
for code in stock_codes:
    name = stocks_info[code]['name']
    data = download_stock_data(code, name, start_date, end_date, is_index=False)
    if data is not None:
        stock_data[code] = data

print(f"\n数据下载完成，成功获取 {len(stock_data)} 只股票数据")

## 5. 数据整合与对齐

In [ ]:
# 将所有价格数据整合到一个DataFrame中
price_data = hs300_data.copy()
price_data.columns = ['HS300']

# 添加股票数据
for code in stock_codes:
    if code in stock_data:
        price_data = price_data.join(stock_data[code], how='outer')

# 删除缺失值过多的日期
print(f"数据整合前：{len(price_data)} 条记录")
price_data = price_data.dropna(thresh=len(price_data.columns)*0.8)  # 至少80%的数据非空
print(f"清洗后：{len(price_data)} 条记录")

# 前向填充缺失值
price_data = price_data.fillna(method='ffill')

# 显示数据概览
print("\n价格数据概览：")
print(price_data.head())
print(f"\n数据统计信息：")
print(price_data.describe())

## 6. 计算对数收益率

In [ ]:
# 计算对数收益率
returns = np.log(price_data / price_data.shift(1)).dropna()

# 计算超额收益率（相对于无风险利率）
excess_returns = returns - rf_daily

print(f"收益率数据形状：{returns.shape}")
print(f"时间范围：{returns.index.min().strftime('%Y-%m-%d')} 至 {returns.index.max().strftime('%Y-%m-%d')}")

# 显示收益率统计
print("\n日收益率统计（%）：")
print((returns * 100).describe())

## 7. 描述性统计分析

In [ ]:
# 计算详细的描述性统计
def descriptive_stats(returns_data):
    """
    计算收益率的详细描述性统计
    """
    stats_dict = {}
    
    for col in returns_data.columns:
        series = returns_data[col].dropna()
        
        # 基本统计量
        mean_ret = series.mean() * 252  # 年化收益率
        std_ret = series.std() * np.sqrt(252)  # 年化波动率
        skewness = stats.skew(series)  # 偏度
        kurt = stats.kurtosis(series)  # 峰度
        
        # 正态性检验（Jarque-Bera检验）
        jb_stat, jb_pvalue = stats.jarque_bera(series)
        
        stats_dict[col] = {
            '年化收益率(%)': mean_ret * 100,
            '年化波动率(%)': std_ret * 100,
            '偏度': skewness,
            '峰度': kurt,
            'JB统计量': jb_stat,
            'JB_P值': jb_pvalue
        }
    
    return pd.DataFrame(stats_dict).T

# 生成描述性统计表
desc_stats = descriptive_stats(returns)

# 添加股票名称
desc_stats['股票名称'] = ['沪深300'] + [stocks_info[code]['name'] for code in stock_codes if code in returns.columns]
desc_stats = desc_stats[['股票名称'] + [col for col in desc_stats.columns if col != '股票名称']]

print("=== 描述性统计分析结果 ===")
print(desc_stats.round(4))

## 8. 数据可视化

In [ ]:
# 绘制收益率时序图
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('股票与指数日收益率时序图', fontsize=16)

# 绘制沪深300
axes[0, 0].plot(returns.index, returns['HS300']*100, alpha=0.7, color='red')
axes[0, 0].set_title('沪深300指数', fontsize=12)
axes[0, 0].set_ylabel('日收益率(%)')
axes[0, 0].grid(True, alpha=0.3)

# 绘制各个股票
colors = ['blue', 'green', 'orange', 'purple', 'brown']
plot_positions = [(0,1), (1,0), (1,1), (2,0), (2,1)]

for i, code in enumerate(stock_codes[:5]):
    if code in returns.columns:
        row, col = plot_positions[i]
        axes[row, col].plot(returns.index, returns[code]*100, alpha=0.7, color=colors[i])
        axes[row, col].set_title(f'{stocks_info[code]["name"]} ({code})', fontsize=12)
        axes[row, col].set_ylabel('日收益率(%)')
        axes[row, col].grid(True, alpha=0.3)

# 设置x轴格式
for ax in axes.flat:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.savefig('../output/returns_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 绘制收益率分布直方图
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('收益率分布直方图与正态性检验', fontsize=16)

# 绘制沪深300
axes[0, 0].hist(returns['HS300']*100, bins=50, alpha=0.7, color='red', edgecolor='black')
axes[0, 0].set_title('沪深300指数')
axes[0, 0].set_xlabel('日收益率(%)')
axes[0, 0].set_ylabel('频数')
axes[0, 0].axvline(returns['HS300'].mean()*100, color='darkred', linestyle='--', label='均值')
axes[0, 0].legend()

# 绘制各个股票
plot_positions = [(0,1), (0,2), (1,0), (1,1), (1,2)]

for i, code in enumerate(stock_codes[:5]):
    if code in returns.columns:
        row, col = plot_positions[i]
        axes[row, col].hist(returns[code]*100, bins=50, alpha=0.7, color=colors[i], edgecolor='black')
        axes[row, col].set_title(f'{stocks_info[code]["name"]} ({code})')
        axes[row, col].set_xlabel('日收益率(%)')
        axes[row, col].set_ylabel('频数')
        axes[row, col].axvline(returns[code].mean()*100, color='darkblue', linestyle='--', label='均值')
        axes[row, col].legend()

plt.tight_layout()
plt.savefig('../output/returns_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. 保存数据

In [ ]:
# 保存原始价格数据
price_data.to_csv('../data_raw/stock_prices.csv')
print("原始价格数据已保存到 data_raw/stock_prices.csv")

# 保存收益率数据
returns.to_csv('../data_clean/daily_returns.csv')
print("日收益率数据已保存到 data_clean/daily_returns.csv")

# 保存超额收益率数据
excess_returns.to_csv('../data_clean/excess_returns.csv')
print("超额收益率数据已保存到 data_clean/excess_returns.csv")

# 保存描述性统计结果
desc_stats.to_csv('../output/descriptive_statistics.csv')
print("描述性统计结果已保存到 output/descriptive_statistics.csv")

# 保存股票基本信息
stock_info_df = pd.DataFrame(stocks_info).T
stock_info_df.index.name = 'code'
stock_info_df.to_csv('../data_clean/stock_info.csv')
print("股票基本信息已保存到 data_clean/stock_info.csv")

## 10. 数据质量检查

In [ ]:
print("=== 数据质量检查 ===")
print(f"价格数据维度：{price_data.shape}")
print(f"收益率数据维度：{returns.shape}")
print(f"数据时间跨度：{(returns.index.max() - returns.index.min()).days} 天")

print("\n各列缺失值情况：")
print(price_data.isnull().sum())

print("\n收益率极端值检查（绝对值大于10%的交易日）：")
for col in returns.columns:
    extreme_count = (abs(returns[col]) > 0.1).sum()
    print(f"{col}: {extreme_count} 天 ({extreme_count/len(returns)*100:.2f}%)")

print("\n数据预处理完成！")